# Lambda 表达式 /匿名函数

## 1.Lambda表达式解决的问题

在 C++ 中，Lambda 表达式就是匿名函数，它是 C++11 引入的核心语法糖，用于在需要函数的地方快速定义一个临时、短小的逻辑单元，无需像普通函数那样单独声明。

In [ ]:
#include<iostream>
#include<vector>
#include<algorithm>
#include<string>
#include<functional>

using namespace std;

class MyPrint
{
public:
    void operator()(int val)
    {
        cout << val+100 << " ";
    }
};

void test01()
{
    vector<int>v1;
    for (int i = 0; i < 10; i++)
    {
        v1.push_back(i);
    }
    vector<int>v2;
    v2.resize(v1.size());
    copy(v1.begin(), v1.end(), v2.begin());

    // 用lambda表达式替代独立的myPrint函数
    //for_each(v2.begin(), v2.end(), MyPrint());
    for_each(v2.begin(), v2.end(), [](int val) { cout << val << " "; });
    cout << endl;
}

int main()
{
    test01();
    return 0;
}

## 2.Lambda 表达式的完整语法结构

一个完整的 Lambda 表达式由 5 部分组成，其中只有 捕获列表 和 函数体 是必须的：

In [ ]:
[捕获列表](参数列表) mutable/noexcept -> 返回类型 
{
    函数体
};

- 1.[] 捕获列表 (Capture List)：
  
    作用：Lambda 是一个闭包（Closure），捕获列表决定了它能否 “看到” 并使用外部（父作用域，如 test01 函数内）的变量。

    你的代码：[] 为空，表示不捕获任何外部变量，只使用传入的参数 val 和全局的 cout。
- 2.(int val) 参数列表 (Parameter List)：
  
    作用：和普通函数的参数列表完全一致。

    你的代码：接收一个 int 类型的参数，对应 for_each 传进来的容器元素。

    C++14 进阶：可以用 auto 让编译器自动推导类型：[](auto val) { ... }。
- 3.-> 返回类型 (Trailing Return Type)：
  
    作用：显式指定返回值类型。

    省略规则：如果函数体只有 return 语句，或者返回 void，编译器可以自动推导，通常可以省略不写。
- 4.{ ... } 函数体 (Body)：
  
    作用：具体执行的代码逻辑。

## 3.核心难点：捕获列表的详细用法
捕获列表是 Lambda 最强大也最容易出错的地方。它控制着 Lambda 以何种方式访问外部变量。

假设有以下外部变量，我们来看各种捕获方式：
```cpp
int a = 10;
int b = 20;
```

#### 3.1 值捕获 `[=]`
*   **语法**：`[=]` 表示把父作用域中所有用到的变量，**以拷贝（值传递）的方式**捕获进来。
*   **特点**：Lambda 内部使用的是变量的“快照”，修改内部值不会影响外部变量。
*   **示例**：
    ```cpp
    auto func = [=]() { 
        cout << a << endl; // 可以读取 a 的值（是 10 的副本）
        // a = 100; // 错误！默认情况下值捕获的变量是只读的（const）
    };
    ```

#### 3.2 引用捕获 `[&]`
*   **语法**：`[&]` 表示把父作用域中所有用到的变量，**以引用的方式**捕获进来。
*   **特点**：Lambda 内部修改值会直接影响外部变量。
*   **⚠️ 风险**：必须确保 Lambda 执行时，外部变量还没有被销毁（生命周期问题）。
*   **示例**：
    ```cpp
    auto func = [&]() { 
        a = 100; // 直接修改外部的 a
        cout << a << endl; 
    };
    ```

#### 3.3 混合捕获（精确控制）
你可以精确指定哪个变量用什么方式捕获：
*   `[a, &b]`：`a` 按值捕获，`b` 按引用捕获。
*   `[=, &b]`：除了 `b` 按引用，其他变量默认按值。
*   `[&, a]`：除了 `a` 按值，其他变量默认按引用。

---

## 4. 进阶关键字：`mutable`
默认情况下，**值捕获**的变量在 Lambda 内部是只读的（`const`）。如果你想在 Lambda 内部修改这个“副本”（注意：改的是副本，不影响外部），需要加 `mutable` 关键字。

```cpp
int a = 10;
// 注意：mutable 必须写在参数列表后面
auto func = [a]() mutable { 
    a = 100; // 正确！修改的是 a 的副本
    cout << "内部: " << a << endl;
};

func();
cout << "外部: " << a << endl; // 外部依然是 10
```

---

## 5. Lambda 的本质（底层原理）
编译器会把 Lambda 表达式自动生成一个**匿名的类（Functor，函数对象）**。

*   **捕获列表** 对应类的**成员变量**。
*   **参数列表** 对应类的 `operator()` 重载函数的参数。

所以，Lambda 只是让你少写了很多代码，编译器在后台帮你生成了一个类。

---

## 6.如何存储 Lambda
由于 Lambda 的类型是编译器生成的匿名类型，你通常有两种方式存储它：

1.  **使用 `auto`（推荐）**：
    ```cpp
    auto my_lambda = [](int x) { return x * 2; };
    ```

2.  **使用 `std::function`（用于函数参数传递或类型擦除）**：
    ```cpp
    #include <functional>
    std::function<int(int)> my_func = [](int x) { return x * 2; };
    ```

---

## 7.典型应用场景
1.  **配合 STL 算法**（如你的代码）：
    替代繁琐的独立函数，让代码逻辑紧凑（`for_each`, `sort`, `find_if` 等）。
2.  **作为回调函数**：
    用于事件处理或异步任务。
3.  **实现闭包（Closure）**：
    利用捕获列表保存状态，实现类似“带状态的函数”的功能。

